In [1]:
# import required python modules
import os
import openpyxl
import pandas as pd

In [1]:
# set input and output paths
base_dir = r"D:\Egypt-Opportunity-Radar-Task3\data\raw\production"
in_path = os.path.join(base_dir, "task3_capmas_private_industrial_production_2021.xlsx")
out_dir = os.path.join(base_dir, "cleaned")
os.makedirs(out_dir, exist_ok=True)

In [1]:
# load raw excel workbook
wb = openpyxl.load_workbook(in_path, data_only=True)
summary_rows = []
cleaned_sheets = {}

In [1]:
# clean each worksheet table
for name in wb.sheetnames:
    ws = wb[name]
    img_count = len(getattr(ws, "_images", []))
    data = ws.values
    df = pd.DataFrame(data)
    df = df.dropna(how="all").dropna(axis=1, how="all")
    df = df.fillna("")
    note = "picture and metadata" if img_count > 0 else "single table sheet"
    summary_rows.append({"sheet_name": name, "rows": len(df), "cols": df.shape[1], "images": img_count, "type": note})
    cleaned_sheets[name[:31]] = df

In [1]:
# extract main production table
t2_raw = cleaned_sheets["Table جدول 2"]
t2_rows = []
for _, row in t2_raw.iterrows():
    vals = [str(v).strip() for v in row.values]
    if len(vals) >= 13 and vals[2].isdigit() and len(vals[2]) == 4:
        t2_rows.append({
            "division": vals[0],
            "group": vals[1],
            "isic_class": vals[2],
            "activity_ar": vals[3],
            "activity_en": vals[12],
            "finished_goods_k_egp": pd.to_numeric(vals[4], errors="coerce"),
            "stock_change_k_egp": pd.to_numeric(vals[5], errors="coerce"),
            "work_for_others_k_egp": pd.to_numeric(vals[6], errors="coerce"),
            "other_revenues_k_egp": pd.to_numeric(vals[7], errors="coerce"),
            "total_production_k_egp": pd.to_numeric(vals[8], errors="coerce"),
            "domestic_sales_k_egp": pd.to_numeric(vals[9], errors="coerce"),
            "export_sales_k_egp": pd.to_numeric(vals[10], errors="coerce")
        })
df_t2 = pd.DataFrame(t2_rows)

In [1]:
# save cleaned outputs
csv_out = os.path.join(out_dir, "cleaned_task3_capmas_private_industrial_production_2021_table2.csv")
df_t2.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [1]:
xlsx_out = os.path.join(out_dir, "cleaned_task3_capmas_private_industrial_production_2021.xlsx")
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="sheet_summary", index=False)
    df_t2.to_excel(writer, sheet_name="cleaned_table_2", index=False)
    for sname, sdf in cleaned_sheets.items():
        sdf.to_excel(writer, sheet_name=sname, index=False, header=False)